# susi_2021 validation figures

Compares the following from a `susi_2021.py` run against the field measurements behind the 2021 SUSI paper:

- **WT time series**: measured and simulated water table, one panel per site.
- **WT, biomass, volume and CO₂**: observed against predicted, 2×2 panel.
- **Residual table**: volume growth residual per site.

**How to reproduce this:**
Run `susi_2021.py`.
Paths, the run ID, the site labels and every per-site value come from
`parameters_2021.py`; nothing per-site is defined here.


In [ ]:
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from netCDF4 import Dataset

from susi.io.project_layout import run_dir
from parameters_2021 import (
    MEASUREMENTS_DIR,
    MEASUREMENTS_PER_SITE,
    PARAMS_PER_SITE,
    PROJECT_DIR,
    RUN_ID,
    SITE_LABELS,
    SiteLabel,
    SiteMeasurements,
    assign_susi_params_to_site,
    get_scenario_label_from_site_label,
    get_stand_label_from_site_label,
)

sns.set()

## Setup

In [ ]:
# Plot style
SUBPANEL_FONTSIZE = 18
AXIS_LABEL_FONTSIZE = 16
TICK_FONTSIZE = 11.0
SHADE_COLOR1 = "yellow"
SHADE_COLOR2 = "grey"
SHADE_ALPHA = 0.3
REGRESSION_LINESTYLE = "k--"
REGRESSION_LINEWIDTH = 2
ERRORBAR_CAPSIZE = 4
SCATTER_MARKERSIZE = 10

In [ ]:
# Measured WT workbooks, one per measurement area.
MEASUREMENTS_FOLDER = MEASUREMENTS_DIR / "Pohjavesiaineistot"
# Measured biomass growth per site.
BIO_FILEPATH = MEASUREMENTS_DIR / "gr_bio.xlsx"
# The run susi_2021.py writes: outputs/<run_id>/<stand>/<scenario>/susi.nc
RUN_FOLDER = run_dir(PROJECT_DIR, RUN_ID)

## Helpers

### Loading the simulation (netcdf)

In [ ]:
def get_netcdf_path_from_site_label(site_label: SiteLabel, run_folder: Path) -> Path:
    stand_label = get_stand_label_from_site_label(site_label)
    scenario_label = get_scenario_label_from_site_label(site_label)

    return run_folder / stand_label / scenario_label / "susi.nc"


@contextmanager
def open_susi_netcdf(site_label: SiteLabel, run_folder: Path):
    ncf = Dataset(
        get_netcdf_path_from_site_label(site_label=site_label, run_folder=run_folder),
        mode="r",
    )
    try:
        yield ncf
    finally:
        ncf.close()


def load_simulation_wt(
    site_label: SiteLabel, run_folder: Path
) -> tuple[pd.DataFrame, datetime, datetime]:
    # Daily WT, one column per computation node (the ditch nodes left out).
    site_params = assign_susi_params_to_site(site_label)
    start_date = site_params.simulation_config.start_date
    end_date = site_params.simulation_config.end_date
    with open_susi_netcdf(site_label, run_folder) as ncf:
        dwt = ncf["strip"]["dwt"][0, :, 1:-1]
        days, cols = np.shape(dwt)
        dfsim = pd.DataFrame(
            dwt, columns=range(cols), index=pd.date_range(start_date, periods=days)
        )
    return dfsim, start_date, end_date


def load_stand_growth(
    site_label: SiteLabel, run_folder: Path
) -> tuple[float, float, float, float]:
    # Mean and std across nodes of the per-node mean yearly growth: volume
    # (m3/ha/yr) and biomass (kg/ha/yr).
    with open_susi_netcdf(site_label, run_folder) as ncf:
        growth = np.array(ncf["stand"]["volumegrowth"][0, :, 1:-1])
        _, ncols = np.shape(growth)
        dfgrowth = pd.DataFrame(growth, columns=range(ncols))

        bm_growth = np.array(
            ncf["stand"]["dominant"]["NPP"][0, :, 1:-1]
            * ncf["stand"]["stems"][0, :, 1:-1]
        )
        dfbm = pd.DataFrame(bm_growth, columns=range(ncols))

    gro = np.mean(dfgrowth.mean(axis=0).values)
    grosd = np.std(dfgrowth.mean(axis=0).values)
    bm_gr = np.mean(dfbm.mean(axis=0).values)
    bm_gr_sd = np.std(dfbm.mean(axis=0).values)
    return gro, grosd, bm_gr, bm_gr_sd


def load_co2_data(
    site_label: SiteLabel, run_folder: Path
) -> tuple[
    np.ndarray, int, pd.DataFrame, pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray
]:
    # Yearly soil CO2 per node from ESOM, and the Ojanen 2010 empirical model
    # driven by the same run's volume, growing-season WT and peat temperature.
    # Row 0 (the initial state) is left at zero in `empirical`.
    site_params = assign_susi_params_to_site(site_label)
    sday = site_params.simulation_config.start_date
    end_date = site_params.simulation_config.end_date
    sfc = site_params.stand_params.site_fertility_class
    with open_susi_netcdf(site_label, run_folder) as ncf:
        vol = np.array(ncf["stand"]["volume"][0, :, 1:-1])
        _, cols = np.shape(vol)
        dfvol = pd.DataFrame(vol, columns=range(cols))

        dwt = ncf["strip"]["dwtyr_growingseason"][0, :, 1:-1]
        yrs, cols = np.shape(dwt)
        dfwt = pd.DataFrame(dwt, columns=range(cols))

        peat_t = ncf["temperature"]["T"][0, :, 3]
        days = np.shape(peat_t)[0]
        dft = pd.DataFrame(
            peat_t, columns=pd.Index(["T"]), index=pd.date_range(sday, periods=days)
        )

        esom_co2 = ncf["esom"]["Mass"]["co2"][0, :, 1:-1] / 10.0

    empirical = np.zeros(np.shape(dwt))
    for i, yr in enumerate(range(sday.year, end_date.year + 1)):
        t_peat_yr = np.ravel(dft.loc[str(yr)].values)
        rhet = ojanen_2010(sfc, vol[i + 1], t_peat_yr, dfwt.loc[i + 1].values * -100.0)
        empirical[i + 1, :] = rhet

    return vol, yrs, dfvol, dfwt, dft, esom_co2, empirical

### Loading the measurements

In [ ]:
def load_wt_measurements(
    site_measurements: SiteMeasurements, measurements_folder: Path
) -> pd.DataFrame:
    # Measured WT of the site's tubes, in m, negative below the surface.
    file_meas = site_measurements.file
    tubes = site_measurements.tubes
    dfmeas = pd.read_excel(measurements_folder / file_meas, sheet_name="CSV")
    dfmeas["date"] = pd.to_datetime(
        dict(year=dfmeas.vuosi, month=dfmeas.kk, day=dfmeas.pv)
    )
    dfmeas = dfmeas.set_index("date")
    dfmeas = dfmeas.drop(["vuosi", "kk", "pv"], axis=1)
    dfmeas = dfmeas.drop(columns=[col for col in dfmeas if col not in tubes])
    dfmeas = dfmeas / 100.0 * -1
    return dfmeas

### Ojanen 2010 empirical CO₂ model

Heterotrophic soil respiration from stand volume, bulk density (by site fertility class),
growing-season WT and peat temperature.

In [ ]:
def ojanen_2010(sfc, stand_v, t_peat, gs_wt):
    # Mese study: bulk densities in different fertility classes, g/cm3
    bd_d = {
        2: 0.14,
        3: 0.11,
        4: 0.10,
        5: 0.08,
    }
    # Bulk density by site fertility class, kg/m3
    bd = bd_d[sfc] * 1000.0
    B = 350.0
    T5zero = -46.02
    T5ref = 10.0
    Rref = (
        0.0695
        + 3.7 * 10 ** (-4) * stand_v
        + 5.4 * 10 ** (-4) * bd
        + 1.2 * 10 ** (-3) * gs_wt
    ) * 24.0  # unit g/m2/h CO2-> g/m2/day
    Rhet = [
        rref * np.exp(B * ((1.0 / (T5ref - T5zero)) - (1.0 / (t_peat - T5zero))))
        for rref in Rref
    ]
    Rhet = np.array(Rhet).T
    return np.sum(Rhet, axis=0)

### Plotting utilities

In [ ]:
def add_quadrant_shading(x0: float, x1: float) -> None:
    # Shades above the 1:1 line in one colour and below it in another.
    plt.fill_between(
        [x0, x1], [x1, x1], [x0, x1], color=SHADE_COLOR1, alpha=SHADE_ALPHA
    )
    plt.fill_between(
        [x0, x1], [x0, x1], [x0, x0], color=SHADE_COLOR2, alpha=SHADE_ALPHA
    )


def add_subplot_label(ax, label: str) -> None:
    ax.text(
        0.04,
        0.95,
        label,
        horizontalalignment="left",
        verticalalignment="top",
        fontsize=SUBPANEL_FONTSIZE,
        transform=ax.transAxes,
        fontweight="bold",
    )


def fit_regression(obs: np.ndarray, sim: np.ndarray) -> tuple[float, str]:
    # Least-squares slope of sim = a * obs, through the origin.
    a, _, _, _ = np.linalg.lstsq(obs[:, np.newaxis], sim, rcond=None)
    eq = "y = " + str(np.round(a[0], 3)) + "x"
    return a[0], eq


def plot_regression_line(x_range: np.ndarray, a: float) -> None:
    plt.plot(x_range, a * x_range, REGRESSION_LINESTYLE, linewidth=REGRESSION_LINEWIDTH)


def plot_scatter_site(
    x: np.ndarray,
    y: np.ndarray,
    yerr: np.ndarray,
    xerr: np.ndarray | None = None,
    label: str = "",
    color: str | None = None,
    capsize: int = ERRORBAR_CAPSIZE,
    markersize: int = SCATTER_MARKERSIZE,
) -> None:
    # Error bars span two standard deviations.
    plt.plot(x, y, "o", markersize=markersize, label=label, color=color)
    if xerr is not None:
        plt.errorbar(x, y, yerr * 2, xerr * 2, "none", color=color, capsize=capsize)
    else:
        plt.errorbar(x, y, yerr * 2, 0, "none", color=color, capsize=capsize)

## Figure 1: WT time series

Measured WT (blue dots) against the range of the simulated WT across the strip's nodes (grey band).

In [ ]:
COLS = 2
ROWS = 6

fig = plt.figure(figsize=(10.0, 15.0))
gs = gridspec.GridSpec(ROWS, COLS, figure=fig)
# Two panels per row, in SITE_LABELS order; the last row has one.
coordinates = [gs[i // COLS, i % COLS] for i in range(len(SITE_LABELS))]
abc = "abcdefghijk"

assert len(SITE_LABELS) == len(abc)
assert len(SITE_LABELS) <= ROWS * COLS

for i, (crd, site_label, tx) in enumerate(zip(coordinates, SITE_LABELS, abc)):
    dfmeas = load_wt_measurements(
        MEASUREMENTS_PER_SITE[site_label], MEASUREMENTS_FOLDER
    )

    dfsim, sday, end_date = load_simulation_wt(site_label, RUN_FOLDER)

    wt_max = np.max(dfsim, axis=1)
    wt_min = np.min(dfsim, axis=1)
    ax = fig.add_subplot(crd)
    ax.fill_between(dfsim.index, wt_max, wt_min, color="grey", alpha=0.3)
    ax.set_ylim((-1.0, 0.0))
    ax.plot(dfmeas, "bo", markersize=2)
    ax.set_xlim((mdates.date2num(sday), mdates.date2num(end_date)))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    site_name = MEASUREMENTS_PER_SITE[site_label].name

    ax.text(
        0.02,
        0.95,
        tx + "  " + site_name,
        horizontalalignment="left",
        verticalalignment="top",
        fontsize=14,
        transform=ax.transAxes,
    )

    for xlabel_i in ax.get_xticklabels():
        xlabel_i.set_fontsize(TICK_FONTSIZE)
        xlabel_i.set_y(-0.01)

    # Right-hand column: no y tick labels.
    if i in [1, 3, 5, 7, 9]:
        for ylabel_i in ax.get_yticklabels():
            ylabel_i.set_fontsize(0.0)
            ylabel_i.set_visible(False)
    else:
        for ylabel_i in ax.get_yticklabels():
            ylabel_i.set_fontsize(TICK_FONTSIZE)
            ylabel_i.set_x(-0.025)

        ax.set_ylabel("WT, m", fontsize=14)

    # Bottom panels: x axis label.
    if i in [9, 10]:
        ax.set_xlabel("Time", fontsize=14)
plt.tight_layout(h_pad=1.5)
plt.show()

## Figure 2: WT, biomass, volume and CO₂

### Late-summer WT, measured and simulated

Per site and year: the mean and std across tubes (measured) or nodes (simulated) of each one's
July–August mean WT.

In [ ]:
out = {}
for site_label in SITE_LABELS:
    dfmeas = load_wt_measurements(
        MEASUREMENTS_PER_SITE[site_label], MEASUREMENTS_FOLDER
    )

    dfsim, start_date, end_date = load_simulation_wt(site_label, RUN_FOLDER)

    print(site_label, start_date, end_date)
    dfmeas = dfmeas.sort_index()[str(dfsim.index[0]) : str(dfsim.index[-1])]

    out[site_label] = {}
    meansim = []
    stdsim = []
    meanmeas = []
    stdmeas = []
    for yr in range(start_date.year, end_date.year + 1):
        wtsim = dfsim[str(yr) + "-07-01" : str(yr) + "-08-31"].mean().values
        meansim.append(wtsim.mean())
        stdsim.append(wtsim.std())
        wtmeas = dfmeas[str(yr) + "-07-01" : str(yr) + "-08-31"].mean().values
        meanmeas.append(wtmeas.mean())
        stdmeas.append(wtmeas.std())
    out[site_label]["meansim"] = meansim
    out[site_label]["stdsim"] = stdsim
    out[site_label]["meanmeas"] = meanmeas
    out[site_label]["stdmeas"] = stdmeas

### Volume and biomass growth, simulated and observed

Simulated growth from the netcdf files. Observed volume growth from the measured start and end
volume in `PARAMS_PER_SITE`, where the original read them too; observed biomass growth from
`gr_bio.xlsx`.

In [ ]:
gro = np.zeros(len(SITE_LABELS))
grosd = np.zeros(len(SITE_LABELS))
bm_gr = np.zeros(len(SITE_LABELS))
bm_gr_sd = np.zeros(len(SITE_LABELS))

for i, site_label in enumerate(SITE_LABELS):
    gro[i], grosd[i], bm_gr[i], bm_gr_sd[i] = load_stand_growth(site_label, RUN_FOLDER)

dvol = {
    "sites": SITE_LABELS,
    "grsim": gro,
    "grsd": grosd,
    "bmgr": bm_gr,
    "bmgrsd": bm_gr_sd,
}
dfvols = pd.DataFrame(data=dvol)
dfvols = dfvols.set_index("sites")

print(dfvols)

In [ ]:
dfbio = pd.read_excel(BIO_FILEPATH, index_col="site")

dfvols["grobs"] = dfvols["grsim"] * 0.0
dfvols["yrs"] = dfvols["grsim"] * 0.0
dfvols["bioobs"] = dfvols["grsim"] * 0.0

for site in SITE_LABELS:
    site_params = assign_susi_params_to_site(site)
    volume_ini, volume_end = PARAMS_PER_SITE[site].vol
    print(
        site,
        np.round(volume_end - volume_ini),
        dfvols.loc[site]["grsim"],
    )
    # Observed volume growth over the whole period, m3/ha.
    dfvols.at[site, "grobs"] = volume_end - volume_ini
    # Simulated years, counting both ends.
    dfvols.at[site, "yrs"] = (
        site_params.simulation_config.end_date.year
        - site_params.simulation_config.start_date.year
        + 1.0
    )
    dfvols.at[site, "bioobs"] = dfbio.at[site, "gr_bio"]

### The figure

In [ ]:
fig = plt.figure(num="growth", figsize=(10, 10))
gs = gridspec.GridSpec(ncols=2, nrows=2, figure=fig, wspace=0.25, hspace=0.25)

colors = plt.colormaps["jet"](np.linspace(0, 1, len(SITE_LABELS)))

# -----------WT figure ------------------------------------------------

ax0 = fig.add_subplot(gs[0, 0])

add_quadrant_shading(-1.0, 0.0)
si = []
ob = []
for c, site_label in enumerate(SITE_LABELS):
    site_name = MEASUREMENTS_PER_SITE[site_label].name
    data = out[site_label]
    obs = np.array(data["meanmeas"])
    ob.extend(obs)
    obserr = np.array(data["stdmeas"])
    pre = np.array(data["meansim"])
    si.extend(pre)
    preerr = np.array(data["stdsim"])
    plot_scatter_site(obs, pre, preerr, xerr=obserr, label=site_name, color=colors[c])
si = np.array(si)
ob = np.array(ob)

# Years without measurements are NaN; left out of the RMSE and the fit.
diff = ob - si
diff = diff[~np.isnan(diff)]
rmse = np.round(np.sqrt(np.square(diff).mean()), 3)

dfdata = pd.DataFrame.from_dict({"meas": ob, "sim": si})
dfdata = dfdata.dropna()

a_wt, eq_wt = fit_regression(dfdata["meas"].values, dfdata["sim"].values)
x_wt = np.arange(-0.9, -0.03, 0.05)
plot_regression_line(x_wt, a_wt)

plt.xlim([-1.0, 0.0])
plt.ylim([-1.0, 0.0])
plt.text(-0.9, -0.1, eq_wt, fontsize=AXIS_LABEL_FONTSIZE - 1)
plt.text(-0.9, -0.15, "RMSE " + str(rmse), fontsize=AXIS_LABEL_FONTSIZE - 1)
plt.xlabel(r"Observed $\it{WT}$, m", fontsize=AXIS_LABEL_FONTSIZE)
plt.ylabel(r"Predicted $\it{WT}$, m", fontsize=AXIS_LABEL_FONTSIZE)

add_subplot_label(ax0, "a")

# ----------BM figure ---------------------------------------------------
ax1 = fig.add_subplot(gs[0, 1])

mval = 10000.0
add_quadrant_shading(0.0, mval)

for c, site_label in enumerate(SITE_LABELS):
    site_name = MEASUREMENTS_PER_SITE[site_label].name
    obs = dfvols.loc[site_label]["bioobs"]
    pre = dfvols.loc[site_label]["bmgr"]
    preerr = dfvols.loc[site_label]["bmgrsd"]
    plot_scatter_site(obs, pre, preerr, label=site_name, color=colors[c])
plt.xlabel(r"Observed bm growth, $kg \ ha^{-1} yr^{-1}$ ", fontsize=AXIS_LABEL_FONTSIZE)
plt.ylabel(
    r"Predicted bm growth, $kg \ ha^{-1} yr^{-1}$ ",
    fontsize=AXIS_LABEL_FONTSIZE,
    labelpad=-7.5,
)
plt.xlim([0, mval])
plt.ylim([0, mval])
rmse = np.round(
    np.sqrt(
        np.square(dfvols["bioobs"].values[:, np.newaxis] - dfvols["bmgr"].values).mean()
    ),
    0,
)
a_bm, eq_bm = fit_regression(dfvols["bioobs"].values, dfvols["bmgr"].values)
x_bm = np.arange(1000.0, 9000.0, 100)
plot_regression_line(x_bm, a_bm)
plt.text(5000, 8300, "RMSE " + str(rmse), fontsize=AXIS_LABEL_FONTSIZE - 1)
plt.text(5000, 8900, eq_bm, fontsize=AXIS_LABEL_FONTSIZE - 1)

add_subplot_label(ax1, "b")

# -------------------Vol figure ------------------
ax2 = fig.add_subplot(gs[1, 0])

mval = 12.0
add_quadrant_shading(0.0, mval)

for c, site_label in enumerate(SITE_LABELS):
    site_name = MEASUREMENTS_PER_SITE[site_label].name
    obs = dfvols.loc[site_label]["grobs"] / dfvols.loc[site_label]["yrs"]
    pre = dfvols.loc[site_label]["grsim"]
    preerr = dfvols.loc[site_label]["grsd"]
    plot_scatter_site(obs, pre, preerr, label=site_name, color=colors[c])

# One legend for the whole figure, above panels a and b.
ax1.legend(
    loc="upper center",
    bbox_to_anchor=(-0.15, 1.3),
    ncol=4,
    fontsize=AXIS_LABEL_FONTSIZE - 3,
)
plt.xlim([0, mval])
plt.ylim([0, mval])

plt.xlabel(
    r"Observed $\it{i_V}$, $m^{3} ha^{-1} yr^{-1}$ ", fontsize=AXIS_LABEL_FONTSIZE
)
plt.ylabel(
    r"Predicted $\it{i_V}$, $m^{3} ha^{-1} yr^{-1}$ ", fontsize=AXIS_LABEL_FONTSIZE
)

obs_growths = dfvols["grobs"].values / dfvols["yrs"].values
sim_growths = dfvols["grsim"].values

diffv = obs_growths - sim_growths
rmse = np.round(np.sqrt(np.square(diffv).mean()), 3)
a_v, eq_v = fit_regression(obs_growths, sim_growths)

x_v = np.arange(1.0, 8.0, 0.5)
plt.text(6, 8.3, "RMSE " + str(rmse), fontsize=AXIS_LABEL_FONTSIZE - 1)
plot_regression_line(x_v, a_v)
plt.text(6, 9, eq_v, fontsize=AXIS_LABEL_FONTSIZE - 1)

add_subplot_label(ax2, "c")

# -------------------CO2 figure ------------------
ax3 = fig.add_subplot(gs[1, 1])

mval = 3000.0
add_quadrant_shading(0.0, mval)
esarr = np.empty(0)
emps = np.empty(0)
for c, site_label in enumerate(SITE_LABELS):
    vol, yrs, dfvol, dfwt, dft, esom_co2, empirical = load_co2_data(
        site_label, RUN_FOLDER
    )

    # Printed only: the Ojanen 2019 soil CO2 balance and the soil C balance
    # (litter in minus CO2-C out) as CO2, for comparison.
    with open_susi_netcdf(site_label, RUN_FOLDER) as ncf:
        ojanen_2019 = np.mean(ncf["ojanen"]["soil_co2_balance"][0, :, 1:-1])
        soil_bal = (
            np.mean(
                ncf["balance"]["C"]["stand_litter_in"][0, :, 1:-1]
                + ncf["balance"]["C"]["gv_litter_in"][0, :, 1:-1]
                - ncf["balance"]["C"]["co2c_release"][0, :, 1:-1]
            )
            * 44
            / 12
        )

        print(site_label, ojanen_2019, soil_bal)
    # One line per simulated year, across the nodes; year 0 is the initial state.
    for m in range(1, yrs):
        plt.plot(empirical[m, :], esom_co2[m, :], color=colors[c])

    esarr = np.append(esarr, np.ravel(esom_co2[1:, :]))
    emps = np.append(emps, np.ravel(empirical[1:, :]))

plt.xlim([0, mval])
plt.ylim([0, mval])

plt.xlabel(r"Empirical $CO_2$, $g m^{-2} yr^{-1}$ ", fontsize=AXIS_LABEL_FONTSIZE)
plt.ylabel(
    r"Esom $CO_2$, $g m^{-3} yr^{-2}$ ", fontsize=AXIS_LABEL_FONTSIZE, labelpad=-5.0
)

a_c, eq_c = fit_regression(emps, esarr)
x_c = np.arange(200.0, 2000.0, 0.5)
plot_regression_line(x_c, a_c)
plt.text(2000, 2000, eq_c, fontsize=AXIS_LABEL_FONTSIZE - 1)

add_subplot_label(ax3, "d")

plt.show()

## Residual table

Mean yearly volume growth per site, observed and predicted (the same values as panel c), and
`pre - obs`.

In [ ]:
dfresid = pd.DataFrame(
    {
        "name": [MEASUREMENTS_PER_SITE[site_label].name for site_label in SITE_LABELS],
        # The site fertility class that was simulated. The original's
        # hand-written list said 5 for parkano11; the simulation uses 3.
        "sfc": [PARAMS_PER_SITE[site_label].sfc for site_label in SITE_LABELS],
        "obs": (dfvols["grobs"] / dfvols["yrs"]).values,
        "pre": dfvols["grsim"].values,
    }
)
dfresid["residual"] = dfresid["pre"] - dfresid["obs"]
print(dfresid)